In [ ]:
# When analyzing from scratch, first run in FIJI macro RNAi_fiji_manual_analysis_1.ijm and then analysis_2 and analysis_3

# This script needs the files ending in mask_GFP_lobula.tif, mask_GFP_medulla.tif, mask_lobula.tif and mask_medulla.tif
# also medulla_manual_mask_medulla_GFP.tif and lobula_manual_mask_lobula_GFP.tif
# of each .lif file that has a corresponding .tif of three middle slices. These files are created when running the FIJI macros.

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import seaborn as sns
import pandas as pd
import os
import pathlib 
import re

style_file = pathlib.Path.cwd().parent / "plot_style.mplstyle"
plt.style.use(["default", style_file])

inputpath_data = pathlib.Path.cwd() / 'input_data/staining_RNAi_efficacy' 
inputpath_data.mkdir(parents=True, exist_ok=True)
savepath_data = pathlib.Path.cwd() / 'output_data/staining_RNAi_efficacy' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig/staining_RNAi_efficacy' 
savepath_fig.mkdir(parents=True, exist_ok=True)

cm_inch = 1/2.54

%config InlineBackend.figure_format = 'retina'

In [ ]:
palette_genotypes = {
                     'HisCl1_double_RNAi': '#828288', 
                     'His_HisCl1_double_RNAi': '#828288', 
                     'ort_RNAi': '#e8872c',
                     'RY_Rdl_RNAi': '#52c3c3',
}

genotype_formatting = {'HisCl1_double_RNAi': 'HisCl1 RNAi', 
                     'His_HisCl1_double_RNAi': 'HisCl1 RNAi',
                     'ort_RNAi': 'Ort RNAi',
                     'RY_Rdl_RNAi': 'Rdl RNAi', 
}


In [ ]:
# Collect files and file names
# CURRENT DIRECTORY: sample/analysis (save it in the analysis folder so that you don't have to adapt the directory)

files_list = []

for path, subdirs, files in os.walk(inputpath_data):
    for name in files:
        if not name.startswith('.') and name.endswith('.lif') and name.startswith('20'):
            name = pathlib.Path(name)
            files_list.append(name.stem)


In [ ]:
# Function to extract the information of the genotype, brain, etc. 
def extract_part(filename, part):
    
    if part == 'genotype':
        match = re.search(r'_(BZ\d{3}_)\d+_(.*?)_anti', filename)
        return match.group(2) if match else None

    if part == 'staining':
        match = re.search(r'_(BZ\d{3})_\d+_(.*?)_anti', filename)
        return match.group(1) if match else None
        
    elif part == 'brain':
        # Match 'brain_' and everything after it up to '_channel'
        match = re.search(r'brain_(.*?)_', filename)
        return match.group(1).strip() if match else None
    
    elif part == 'antibody':
        match = re.search(r'anti[-_](.*)_brain', filename)
        antibody_age = match.group(1).strip().lower()
        return antibody_age.split('_')[0] 

    elif part == 'age':
        match = re.search(r'anti[-_](.*)_brain', filename)
        antibody_age = match.group(1).strip().lower()
        if (len(antibody_age.split('_')) == 2):
            age = antibody_age.split('_')[1]
        else:
            age = '3-6d'
        return age 

    return None

In [ ]:
# Create a list to hold the extracted data
data = []

# Extract the required parts
for file in files_list:
    genotype = extract_part(file, "genotype")
    staining = extract_part(file, "staining")
    brain = extract_part(file, "brain")
    antibody = extract_part(file, "antibody")
    age = extract_part(file, "age")

    # Append extracted data as a dictionary
    data.append({
        'file_name': file,
        'staining': staining,
        'genotype': genotype,
        'brain': brain,
        'antibody': antibody,
        'age': age
        })

# Create df from the data list
files_df = pd.DataFrame(data)
files_df

In [ ]:
files_df.groupby(['antibody', 'genotype']).staining.value_counts()

In [ ]:
def open_image(path):
    im = Image.open(path)
    i = 0
    frames = []
    try:
        while True:
            im.seek(i)
            frames.append(np.array(im))
            i += 1
    except EOFError:
        pass
    
    frames = np.array(frames)
    return frames

def make_0_1(mask):
    return np.where(mask == 255, True, False)

def calculate_ratio(path, area):
    # Open all data and corresponding masks
    file_path = pathlib.Path(inputpath_data, path)
    
    data_path = file_path.with_suffix('.tif')
    datafile = open_image(data_path)
    
    # Every .tif file was saved with 3 z-slices from the middle of the stack. Separate the colour channels into separate z-stacks:
    frames_GFP = datafile[[0, 3, 6]]
    frames_receptor = datafile[[1, 4, 7]]
    frames_Brp = datafile[[2, 5, 8]]
    
    GFP_mask_path_medulla = file_path.with_name(file_path.stem + '_mask_GFP_medulla').with_suffix('.tif')
    medulla_GFP_mask = make_0_1(open_image(GFP_mask_path_medulla))

    GFP_mask_path_lobula = file_path.with_name(file_path.stem + '_mask_GFP_lobula').with_suffix('.tif')
    lobula_GFP_mask = make_0_1(open_image(GFP_mask_path_lobula))
    
    medulla_mask_path = file_path.with_name(file_path.stem + '_mask_medulla').with_suffix('.tif')
    medulla_mask = make_0_1(open_image(medulla_mask_path))
    
    lobula_mask_path = file_path.with_name(file_path.stem + '_mask_lobula').with_suffix('.tif')
    lobula_mask = make_0_1(open_image(lobula_mask_path))

    M10_GFP_manual_mask_path = file_path.with_name(file_path.stem + '_mask_GFP_medulla_manual_mask_medulla_GFP').with_suffix('.tif')
    M10_GFP_manual_mask = make_0_1(open_image(M10_GFP_manual_mask_path))

    Lo1_GFP_manual_mask_path = file_path.with_name(file_path.stem + '_mask_GFP_lobula_manual_mask_lobula_GFP').with_suffix('.tif')
    Lo1_GFP_manual_mask = make_0_1(open_image(Lo1_GFP_manual_mask_path))
    
    M10_GFP_mask_medulla = M10_GFP_manual_mask & medulla_GFP_mask
    Lo1_GFP_mask_lobula = Lo1_GFP_manual_mask & lobula_GFP_mask
    
    no_GFP_mask_medulla = np.where(M10_GFP_mask_medulla == 1, False, True)
    no_GFP_mask_lobula = np.where(Lo1_GFP_mask_lobula == 1, False, True)
    medulla_noGFP_mask = medulla_mask & no_GFP_mask_medulla
    lobula_noGFP_mask = lobula_mask & no_GFP_mask_lobula

    # Calculate average receptor intensity in medulla and lobula areas in GFP and noGFP regions, take ratio
    avg_receptor_inside_GFP_medulla = (frames_receptor * M10_GFP_mask_medulla).sum() / M10_GFP_mask_medulla.sum()
    avg_receptor_outside_GFP_medulla = (frames_receptor * medulla_noGFP_mask).sum() / medulla_noGFP_mask.sum() 
    ratio_medulla = avg_receptor_inside_GFP_medulla / avg_receptor_outside_GFP_medulla
    
    avg_receptor_inside_GFP_lobula = (frames_receptor * Lo1_GFP_mask_lobula).sum() / Lo1_GFP_mask_lobula.sum() 
    avg_receptor_outside_GFP_lobula = (frames_receptor * lobula_noGFP_mask).sum() / lobula_noGFP_mask.sum() 
    ratio_lobula = avg_receptor_inside_GFP_lobula / avg_receptor_outside_GFP_lobula
    
    return ratio_medulla if area == 'medulla' else ratio_lobula
    

In [ ]:
files_df = pd.DataFrame(data)
files_df['receptor_ratio_medulla'] = files_df.apply(lambda row: calculate_ratio(row.file_name, 'medulla'), axis=1)
files_df['receptor_ratio_lobula'] = files_df.apply(lambda row: calculate_ratio(row.file_name, 'lobula'), axis=1)

In [ ]:
files_df.groupby(['genotype', 'antibody']).receptor_ratio_medulla.mean()

In [ ]:
average_ratio_medulla = pd.DataFrame(files_df.groupby(['genotype', 'antibody']).receptor_ratio_medulla.mean()).rename({0:'receptor_ratio_medulla'}).reset_index()
average_ratio_lobula = pd.DataFrame(files_df.groupby(['genotype', 'antibody']).receptor_ratio_lobula.mean()).rename({0:'receptor_ratio_lobula'}).reset_index()
average_ratios = average_ratio_medulla
average_ratios['receptor_ratio_lobula'] = average_ratio_lobula.receptor_ratio_lobula

normalization_factors_ort = average_ratios.query("genotype == 'HisCl1_double_RNAi' and antibody == 'ort'")
normalization_factors_ort = normalization_factors_ort[['receptor_ratio_medulla', 'receptor_ratio_lobula']].to_dict(orient='list')

normalization_factors_rdl = average_ratios.query("genotype == 'His_HisCl1_double_RNAi' and antibody == 'rdl'")
normalization_factors_rdl = normalization_factors_rdl[['receptor_ratio_medulla', 'receptor_ratio_lobula']].to_dict(orient='list')
normalization_factors_medulla = {'ort':normalization_factors_ort['receptor_ratio_medulla'][0], 'rdl':normalization_factors_rdl['receptor_ratio_medulla'][0]}
normalization_factors_lobula = {'ort':normalization_factors_ort['receptor_ratio_lobula'][0], 'rdl':normalization_factors_rdl['receptor_ratio_lobula'][0]}

files_df['normalization_factor_medulla'] = files_df['antibody'].map(normalization_factors_medulla)
files_df['normalization_factor_lobula'] = files_df['antibody'].map(normalization_factors_lobula)
files_df['receptor_ratio_medulla_normalized'] = files_df['receptor_ratio_medulla'] / files_df['normalization_factor_medulla'] 
files_df['receptor_ratio_lobula_normalized'] = files_df['receptor_ratio_lobula'] / files_df['normalization_factor_lobula'] 

average_ratio_medulla_normalized = pd.DataFrame(files_df.groupby(['genotype', 'antibody']).receptor_ratio_medulla_normalized.mean()).rename({0:'receptor_ratio_medulla_normalized'}).reset_index()
average_ratio_lobula_normalized = pd.DataFrame(files_df.groupby(['genotype', 'antibody']).receptor_ratio_lobula_normalized.mean()).rename({0:'receptor_ratio_lobula_normalized'}).reset_index()
average_ratios['receptor_ratio_medulla_normalized'] = average_ratio_medulla_normalized.receptor_ratio_medulla_normalized
average_ratios['receptor_ratio_lobula_normalized'] = average_ratio_lobula_normalized.receptor_ratio_lobula_normalized


In [ ]:
fig, ax = plt.subplots(1,4, figsize=(3.5,1.5), dpi=300, sharey=True, squeeze=False)

query = "antibody == 'rdl'"
y_var = 'receptor_ratio_medulla_normalized'
ax_id = ax[0,0]
files_subset = files_df.query(query)
averages_subset = average_ratios.query(query)
sns.stripplot(data=files_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=0.3, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=ax_id, legend=None, order=['His_HisCl1_double_RNAi', 'RY_Rdl_RNAi', 'RO_Rdl_RNAi']);
sns.scatterplot(data=averages_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=1, ax=ax_id, legend=None, zorder=3, marker='_', s=150, lw=1.5);
ax_id.set_title(y_var[15:22] + '\n' + query[13:16])
ax_id.set_ylabel('Norm. receptor intensity ratio (AU)')

query = "antibody == 'rdl'"
y_var = 'receptor_ratio_lobula_normalized'
ax_id = ax[0,1]
files_subset = files_df.query(query)
averages_subset = average_ratios.query(query)
sns.stripplot(data=files_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=0.3, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=ax_id, legend=None, order=['His_HisCl1_double_RNAi', 'RY_Rdl_RNAi', 'RO_Rdl_RNAi']);
sns.scatterplot(data=averages_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=1, ax=ax_id, legend=None, zorder=3, marker='_', s=150, lw=1.5);
ax_id.set_title(y_var[15:21] + '\n' + query[13:16])

query = "antibody == 'ort'"
y_var = 'receptor_ratio_medulla_normalized'
ax_id = ax[0,2]
files_subset = files_df.query(query)
averages_subset = average_ratios.query(query)
sns.stripplot(data=files_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=0.3, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=ax_id, legend=None, order=['HisCl1_double_RNAi', 'ort_RNAi']);
sns.scatterplot(data=averages_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=1, ax=ax_id, legend=None, zorder=3, marker='_', s=150, lw=1.5);
ax_id.set_title(y_var[15:22] + '\n' + query[13:16])

query = "antibody == 'ort'"
y_var = 'receptor_ratio_lobula_normalized'
ax_id = ax[0,3]
files_subset = files_df.query(query)
averages_subset = average_ratios.query(query)
sns.stripplot(data=files_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=0.3, s=3, jitter=0.1, dodge=False, edgecolor='white', linewidth=0.2, ax=ax_id, legend=None, order=['HisCl1_double_RNAi', 'ort_RNAi']);
sns.scatterplot(data=averages_subset, x='genotype', y=y_var, palette=palette_genotypes, hue='genotype', alpha=1, ax=ax_id, legend=None, zorder=3, marker='_', s=150, lw=1.5);
ax_id.set_title(y_var[15:21] + '\n' + query[13:16])

for ax_id in [(0,0), (0,1), (0,2), (0,3)]:
    ax[ax_id].set_ylim([0, 1.2])
    ax[ax_id].set_yticks([0, 0.5, 1])
    ax[ax_id].set_xlabel('');
    ax[ax_id].margins(x=0.5);
    ax[ax_id].spines[['top', 'right']].set_visible(False)

# plt.subplots_adjust(wspace=0.5)

figtitle = "RNAi_intensity_ratios.pdf"
plt.savefig(savepath_fig / figtitle)

In [ ]:
graphpad_file = files_df.sort_values(['antibody', 'genotype', 'brain']).reset_index(drop=True)

graphpad_file_ort_medulla = graphpad_file.query("antibody == 'ort'")[['genotype', 'receptor_ratio_medulla_normalized', 'brain']]
graphpad_file_ort_medulla.pivot(columns='genotype', index='brain', values='receptor_ratio_medulla_normalized').to_csv(savepath_data / "RNAi_normalized_ort_medulla.csv")

graphpad_file_ort_lobula = graphpad_file.query("antibody == 'ort'")[['genotype', 'receptor_ratio_lobula_normalized', 'brain']]
graphpad_file_ort_lobula.pivot(columns='genotype', index='brain', values='receptor_ratio_lobula_normalized').to_csv(savepath_data / "RNAi_normalized_ort_lobula.csv")

graphpad_file_rdl_medulla = graphpad_file.query("antibody == 'rdl'")[['genotype', 'receptor_ratio_medulla_normalized', 'brain']]
graphpad_file_rdl_medulla.pivot(columns='genotype', index='brain', values='receptor_ratio_medulla_normalized').to_csv(savepath_data / "RNAi_normalized_rdl_medulla.csv")

graphpad_file_rdl_lobula = graphpad_file.query("antibody == 'rdl'")[['genotype', 'receptor_ratio_lobula_normalized', 'brain']]
graphpad_file_rdl_lobula.pivot(columns='genotype', index='brain', values='receptor_ratio_lobula_normalized').to_csv(savepath_data / "RNAi_normalized_rdl_lobula.csv")